### **Student Information**
GitHub ID: patrickkosasih

Kaggle name: Patrick K

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

# **Project Report**

## 1. Model Development (10 pts Required)

### 1.1 Preprocessing Steps

We start by joining the tweet text, official split, and emotion labels by `post_id`. This gives us 3,360 labeled training posts and 1,440 test posts. We then trim the surrounding whitespace and check for empty text for all posts.

Then, we split the labeled posts into five folds, stratified by emotion and grouped by cleaned text. This keeps identical tweets together so the same text cannot appear in both the training and validation portions of a fold. We use the first fold for development and all five for the validation audit.

Later, in the SemEval approach, tweets with emotion intensity below 0.35, repeated or conflicting examples, and exact overlaps with either competition split are excluded. This leaves 2,702 auxiliary tweets.

### 1.2 Feature Engineering Steps

We compute word TF-IDF using unigrams and bigrams and character TF-IDF using sequences of three to five characters. For each run, both vocabularies are fitted on the training rows with a minimum document frequency of two. These features are then combined with a normalized 384-dimensional embedding from the frozen (pre-trained and fixed) MiniLM. The TF-IDF features capture specific words and spelling patterns, while the embedding captures information about the whole tweet.

For fine-tuning, we use MiniLM's tokenizer and limit it to 96 tokens to make training faster. The same label order is used for every classifier so their probability columns match.

### 1.3 Explanation of The Model

For each tweet, three classifiers estimate a probability for each of the six emotions. We average their probabilities, giving each classifier the same weight.

1. **Logistic regression** learns how much each input feature should count toward each emotion. Here the inputs are TF-IDF features from words and characters, plus the vector produced by frozen MiniLM. The model adds up the weighted features to give each emotion a score, then converts those scores into probabilities that sum to 1.
2. **MiniLM trained on the competition data** starts from a pretrained language model. We fine-tune it for four epochs, or four passes through the training data, so it learns to predict the competition's emotion labels.
3. **MiniLM trained first on extra emotion examples** starts from the same pretrained model. SemEval is a workshop where researchers compare systems on shared language tasks. The lab's files are stored under `data/semeval` and contain tweets from the related EmoInt emotion-intensity task, with labels for anger, fear, joy, and sadness and ratings for how strongly the emotion is expressed. We use the ratings to filter examples and train MiniLM on the emotion labels for two epochs. We then continue training on the six competition labels for four epochs. [SemEval Affect in Tweets](https://aclanthology.org/S18-1001/) · [EmoInt dataset](https://aclanthology.org/W17-5205/)

Both MiniLM runs use seed 42 for repeatability. The input embedding table, which maps tokens to vectors, is kept frozen during training.

Before picking a label, we divide each averaged score by that emotion's share of the training data raised to the power of 0.25. This gives less common emotions a larger boost. We also multiply the disgust score by 1.10 because the earlier model often labeled disgust as anger. The emotion with the highest adjusted score is selected. In the validation audit, the number of disgust tweets labeled as anger fell from 155 to 140 compared with the original one-seed approach.

Across five validation folds, the final approach averaged **0.5896 macro F1**, compared with **0.5853** for the original one-seed approach in the same audit. Macro F1 gives each emotion equal weight. When the settings were chosen on four folds and evaluated on the fifth, the average was **0.5850**. This suggests that the small improvement may depend on which examples were used to choose the settings.

We refit the selected classifiers using all 3,360 labeled posts and export one emotion per test ID in sample-submission order. The final public Kaggle score was **0.5935**, and private score **0.5944**.


### 1.4 AI Prompts and Techniques Suggested by AI

I used Codex to work through every step. For each step, I used it to guide me and explain concepts, give suggestions on how to improve, experimenting, and implementing the code.

At first, TF-IDF and linear SVM were used as a baseline, and it scored 0.4990 in Kaggle. The first large improvement happened from using Word and character TF-IDF + frozen MiniLM + SVM, where it got 0.5616 on Kaggle. Adding character features to the word TF-IDF baseline lowers the Kaggle score to 0.4930 before we try the hybrid model. After the hybrid model, we try frozen MiniLM embeddings with an SVM and get 0.4687. We then mix the prediction scores from one fine-tuned MiniLM model and a logistic regression classifier, raising the score to 0.5830. Averaging three fine-tuned MiniLM runs in the approach raises it to 0.5881.

The final approach that was used is an equal-weight approach of direct fine-tuned MiniLM, MiniLM fine-tuned after a SemEval warm start, and logistic regression on TF-IDF and frozen MiniLM features. We adjust the scores based on class frequency and give disgust a 10% boost before choosing the final label. This approach scores 0.5935 on Kaggle.

| Approach | Kaggle score |
| --- | ---: |
| Word TF-IDF + linear SVM | 0.4990 |
| Word and character TF-IDF | 0.4930 |
| Word and character TF-IDF + frozen MiniLM + SVM | 0.5616 |
| Frozen MiniLM embeddings + SVM | 0.4687 |
| One-seed fine-tuned MiniLM + logistic regression | 0.5830 |
| Three-seed fine-tuned MiniLM + logistic regression | 0.5881 |
| Direct MiniLM, SemEval-warmed MiniLM, and logistic regression with score adjustments | **0.5935** |

#### My Prompts to Codex, Word by Word

- > what do i do in part 3 `DM2026-Lab2-Homework.ipynb` of the homework?

- > i downloaded the competition data and copied it to kaggle_data. is that all we need?

- > why does the competition report also ask for all of my AI prompts? does that mean our chat here?

- > let's start with the preprocessing steps. what should we do here?

- > i see, let's implement that in the notebook. implement each step in separate cells. before each code cell, create a header and give a brief description in markdown.

- > do we need to do anything else for part 1? if so, let's start part 2. what do we need to do here?

- > why TF-IDF? why not others like miniLM?

- > sure let's try TF-IDF first. implement all the steps you mentioned here. add the markdown like before.

- > btw, what are we exactly trying to achieve here? we train a model that can classify the emotions then submit it to kaggle? is that what we're going to do in part 3?

- > let's move on to part 3 then. what are the steps we need to do?

- > so we train another model separate from the first one we trained? why?

- > why not just use one model?

- > so in the first model, we only use the train data. but on the refit we also use the test data? or how does it work?

- > what are these "validation posts"? are they separate from the training set? or is it just a group of like X percent of the training set?

- > why does that help?

- > do lots of people do this as well when training models? is it because it's already proven to work?

- > i see. alright you can go ahead and implement this. same thing as before for the markdown cells.

- > do i just upload submission.csv to kaggle, or the entire notebook as well?

- > in the submission, the `post_id` column is supposed to be `id`

- > it only scored 0.4990. any ideas on how to improve this? maybe change the vectorizer to miniLM? don't make edits yet.

- > let's implement that. after that, can you find other ways to improve?

- > your previous solution with only word+character TF-IDF lowered the score to 0.4930. are you sure this one is going to work?

- > sure let's implement that

- > could we replace TF-IDF with miniLM or something?

- > so miniLM is the one converting the document into a vector or something, as well as what was it again?

- > what about the one that classifies the new post?

- > so by this you mean only using miniLM for everything?

- > why not use miniLM for the classification model as well?

- > so we fine tune the refit model only?

- > let's make the the code do miniLM + SVM first.

- > don't change the "submission.csv" file name

- > the score dropped to an all-time low 0.4687 bruh. why is this? can you figure out other methods to improve? after you've found out, report it here and edit the notebook directly.

- > continue please

- > nice it went up to 0.5830 in kaggle. we're going on the right track. any other ways to improve? would removing the refitting make it worse or better? or what about using a better LLM like the gemini/groq API or something? don't make the edits yet.

- > what did you do in this approach?

- > so we train 3 different times with the same model but only a different seed?

- > how would that help?

- > what about more than 3 different seeds? like 5?

- > any sources that support this claim?

- > what do we mean by fine tuning the LLM anyway? do we fine tune the parameters of the already pre-trained LLM itself?

- > let's try this first

- > this time it got 0.5881. any other ways to improve?

- > what are semeval and logistic regression? do a thorough final research to find ways on how to improve this, then make your edits on the notebook.

- > what is frozen miniLM anyway? why is it frozen? is it like the opposite of fine-tuned?

---

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

### 1.1 Load and join the competition data

Join the post text, official split, and training labels by `post_id`. Check that every training post has a label and put the test posts in sample submission order.

In [1]:
from pathlib import Path

import pandas as pd

data_dir = Path("kaggle_data")
posts = pd.read_json(data_dir / "final_posts.json")
splits = pd.read_csv(data_dir / "data_identification.csv")
labels = pd.read_csv(data_dir / "emotion.csv")
submission_ids = pd.read_csv(data_dir / "sampleSubmission.csv", usecols=["post_id"])

posts_with_split = splits.merge(posts, on="post_id", how="left", validate="one_to_one")
if len(posts_with_split) != len(posts) or posts_with_split["text"].isna().any():
    raise ValueError("Post IDs in the text data and split file must match.")
if set(posts_with_split["set"]) != {"train", "test"}:
    raise ValueError("The split file must contain only train and test rows.")

train_posts = posts_with_split.loc[posts_with_split["set"].eq("train")].merge(
    labels, on="post_id", how="left", validate="one_to_one"
)
if len(train_posts) != len(labels) or train_posts["label"].isna().any():
    raise ValueError("Every training post must have exactly one emotion label.")

test_posts = posts_with_split.loc[posts_with_split["set"].eq("test")]
if set(test_posts["post_id"]) != set(submission_ids["post_id"]):
    raise ValueError("The official test IDs must match the sample submission.")
test_posts = submission_ids.merge(
    test_posts, on="post_id", how="left", validate="one_to_one"
)

print(f"Labeled training posts: {len(train_posts):,}")
print(f"Test posts: {len(test_posts):,}")

Labeled training posts: 3,360
Test posts: 1,440


### 1.2 Clean the post text

Trim leading and trailing whitespace and check for empty posts. Keep punctuation, emoji, hashtags, and negation words because they can carry emotion.

In [2]:
trimmed_posts = 0
for frame in (train_posts, test_posts):
    cleaned_text = frame["text"].str.strip()
    if cleaned_text.eq("").any():
        raise ValueError("Found a post with empty text after trimming.")
    trimmed_posts += (frame["text"] != cleaned_text).sum()
    frame["text"] = cleaned_text

print(f"Posts with surrounding whitespace removed: {trimmed_posts:,}")

Posts with surrounding whitespace removed: 160


### 1.3 Create the validation split

Split the labeled posts into five folds. Stratify by emotion and group by cleaned text so that identical posts stay in the same fold. Set `validation_fold_number` to an index from 0 through 4 to preview that split. Section 3.3 uses the first fold by default and can evaluate all five.

In [3]:
from sklearn.model_selection import StratifiedGroupKFold

splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
grouped_folds = list(splitter.split(train_posts, train_posts["label"], groups=train_posts["text"]))
validation_fold_number = 0
train_idx, validation_idx = grouped_folds[validation_fold_number]
train_fold = train_posts.iloc[train_idx].copy()
validation_fold = train_posts.iloc[validation_idx].copy()

print(f"Training fold: {len(train_fold):,} posts")
print(f"Validation fold: {len(validation_fold):,} posts")

Training fold: 2,687 posts
Validation fold: 673 posts


## 2. Feature Engineering Steps

### 2.1 Tokenize posts and encode the six labels

Use MiniLM's tokenizer with a 96-token limit for the fine-tuned models. Give the six emotions fixed integer IDs so the same probability column represents the same emotion in all three classifiers.

In [4]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup, set_seed

MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
MAX_TOKENS = 96
BATCH_SIZE = 32
EPOCHS = 4
SEED = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    torch.set_num_threads(min(4, torch.get_num_threads()))
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def tokenize_posts(texts):
    tokens = tokenizer(texts.tolist(), padding="max_length", max_length=MAX_TOKENS,
                       truncation=True, return_tensors="pt")
    return tokens["input_ids"], tokens["attention_mask"]


from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
targets = torch.tensor(label_encoder.fit_transform(train_posts["label"]), dtype=torch.long)
class_names = label_encoder.classes_.tolist()
labeled_tokens = tokenize_posts(train_posts["text"])
print("Emotion labels in prediction order:", class_names)

Emotion labels in prediction order: ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']


### 2.2 Encode the posts with frozen MiniLM

Use the pretrained MiniLM encoder to generate one normalized sentence vector per post. These vectors are inputs to the logistic regression classifier in Section 2.3. The two fine-tuned classifiers in Section 3 each start from their own copy of the pretrained model.

In [5]:
from sentence_transformers import SentenceTransformer

frozen_encoder = SentenceTransformer(MODEL_ID, device=str(device))
labeled_embeddings = frozen_encoder.encode(
    train_posts["text"].tolist(), normalize_embeddings=True, show_progress_bar=False
)
test_embeddings = frozen_encoder.encode(
    test_posts["text"].tolist(), normalize_embeddings=True, show_progress_bar=False
)
del frozen_encoder
print("Frozen labeled embedding matrix:", labeled_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Frozen labeled embedding matrix: (3360, 384)


### 2.3 Combine word, character, and sentence features

Fit word unigrams and bigrams and character sequences of length 3 to 5 on the training rows for each run. Logistic regression combines their TF-IDF weights with the frozen MiniLM vectors and returns a probability for each emotion.

In [6]:
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion

def text_probabilities(fit_texts, fit_embeddings, fit_labels, predict_texts, predict_embeddings):
    features = FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
        ("char", TfidfVectorizer(analyzer="char", ngram_range=(3, 5), min_df=2)),
    ])
    X_fit = hstack([features.fit_transform(fit_texts), fit_embeddings], format="csr")
    X_predict = hstack([features.transform(predict_texts), predict_embeddings], format="csr")
    classifier = LogisticRegression(C=3, max_iter=400).fit(X_fit, fit_labels)
    assert np.array_equal(classifier.classes_, np.arange(len(class_names)))
    return classifier.predict_proba(X_predict)

## 3. Model Implementation Steps

### 3.1 Prepare batches for MiniLM

Group posts of similar lengths and trim padding to the longest post in each batch. Put predictions back in input order so they line up with the labels and submission IDs.

In [7]:
def make_batches(lengths, indices, rng=None):
    # Group similar lengths to avoid running the encoder on extra padding.
    order = lengths[indices]
    if rng is not None:
        order = order + rng.uniform(0, 8, len(indices))
    ordered = indices[np.argsort(order, kind="stable")]
    batches = [ordered[start:start + BATCH_SIZE] for start in range(0, len(ordered), BATCH_SIZE)]
    if rng is not None:
        rng.shuffle(batches)
    return batches

def batch_inputs(tokens, lengths, batch):
    width = int(lengths[batch].max())
    return {"input_ids": tokens[0][batch, :width].to(device),
            "attention_mask": tokens[1][batch, :width].to(device)}

@torch.inference_mode()
def predict_probabilities(model, tokens, indices):
    model.eval()
    lengths = tokens[1].sum(dim=1).numpy()
    probabilities = np.empty((len(indices), model.config.num_labels), dtype=np.float32)
    for positions in make_batches(lengths[indices], np.arange(len(indices))):
        batch = indices[positions]
        logits = model(**batch_inputs(tokens, lengths, batch)).logits
        probabilities[positions] = logits.softmax(dim=1).cpu().numpy()
    return probabilities

### 3.2 Fit MiniLM for emotion classification

The direct model trains on competition labels for four epochs. The transfer model first uses two epochs of SemEval training with intensity ≥0.35, then trains on the competition labels. Both use seed 42 and freeze the input embedding table. All exact text overlaps between the auxiliary corpus and the competition are excluded.

In [8]:
# Start with the four emotions in the lab's SemEval training files.
paths = sorted(Path("data/semeval/train").glob("*-ratings-0to1.train.txt"))
if len(paths) != 4:
    raise ValueError("Expected four supplied SemEval training files.")
aux = pd.concat([pd.read_csv(path, sep="\t", header=None,
                             names=["id", "text", "emotion", "intensity"], quoting=3)
                 for path in paths], ignore_index=True)
aux["text"] = aux["text"].str.strip()
# Remove ambiguous repeated texts and any exact overlap with competition inputs.
ambiguous = aux.groupby("text")["emotion"].transform("nunique").gt(1)
aux = aux.loc[~ambiguous & aux["intensity"].ge(.35) &
              ~aux["text"].isin(set(train_posts["text"]) | set(test_posts["text"]))].drop_duplicates("text")
aux_targets = torch.tensor(label_encoder.transform(aux["emotion"]), dtype=torch.long)
aux_tokens = tokenize_posts(aux["text"])
print(f"Auxiliary SemEval training posts: {len(aux):,}")

def fit_emotion_model(competition_rows, warm_start=False):
    set_seed(SEED)
    rng = np.random.default_rng(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, num_labels=len(class_names), id2label=dict(enumerate(class_names)),
        label2id={label: i for i, label in enumerate(class_names)}
    ).to(device)
    model.base_model.embeddings.requires_grad_(False)
    stages = []
    if warm_start:
        stages.append((aux_tokens, aux_targets, np.arange(len(aux)), 2, 3e-5, 3e-4))
    encoder_lr, head_lr = (3e-5, 3e-4) if warm_start else (4e-5, 4e-4)
    stages.append((labeled_tokens, targets, competition_rows, EPOCHS, encoder_lr, head_lr))

    for tokens, labels, rows, epochs, backbone_lr, classifier_lr in stages:
        optimizer = torch.optim.AdamW([
            {"params": [p for p in model.base_model.parameters() if p.requires_grad], "lr": backbone_lr},
            {"params": model.classifier.parameters(), "lr": classifier_lr},
        ], weight_decay=0.01)
        steps = epochs * ((len(rows) + BATCH_SIZE - 1) // BATCH_SIZE)
        scheduler = get_linear_schedule_with_warmup(optimizer, steps // 10, steps)
        lengths = tokens[1].sum(dim=1).numpy()
        model.train()
        for _ in range(epochs):
            for batch in make_batches(lengths, rows, rng):
                optimizer.zero_grad(set_to_none=True)
                loss = model(**batch_inputs(tokens, lengths, batch), labels=labels[batch].to(device)).loss
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
    return model

Auxiliary SemEval training posts: 2,702


### 3.3 Evaluate the selected models

Evaluate the current model and score adjustments on the first grouped fold by default. Set `AUDIT_ALL_FOLDS` to `True` to evaluate the same settings on all five folds. The earlier comparison of model combinations and score adjustments is summarized below. The full-data refit is performed in Section 3.5.

In [9]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

AUDIT_ALL_FOLDS = False
MODEL_SETUP = 'Both'
DECISION = 'both'

def predict_split(fit_rows, predict_texts, predict_embeddings, predict_tokens, predict_rows):
    lexical = text_probabilities(
        train_posts["text"].iloc[fit_rows], labeled_embeddings[fit_rows], targets[fit_rows].numpy(),
        predict_texts, predict_embeddings,
    )
    if MODEL_SETUP == "Both":
        warm_starts, weights = (False, True), (1/3, 1/3)
        scores = lexical / 3
    elif MODEL_SETUP == "Original":
        warm_starts, weights = (False,), (2/3,)
        scores = lexical / 3
    elif MODEL_SETUP == "SemEval":
        warm_starts, weights = (True,), (0.5,)
        scores = lexical / 2
    else:
        raise ValueError(f"Unknown model setup {MODEL_SETUP}")
    for warm_start, weight in zip(warm_starts, weights):
        model = fit_emotion_model(fit_rows, warm_start)
        scores += weight * predict_probabilities(model, predict_tokens, predict_rows)
        del model
    if not np.isfinite(scores).all():
        raise ValueError("Nonfinite prediction scores")
    return scores

def final_labels(probabilities, fit_rows):
    adjusted = probabilities.copy()
    if DECISION in ("prior", "both"):
        counts = np.bincount(targets[fit_rows].numpy(), minlength=len(class_names))
        adjusted /= (counts / counts.sum()) ** 0.25
    if DECISION in ("disgust", "both"):
        adjusted[:, class_names.index("disgust")] *= 1.10
    return adjusted.argmax(axis=1)

validation_runs = []
for tr, va in (grouped_folds if AUDIT_ALL_FOLDS else grouped_folds[:1]):
    probs = predict_split(tr, train_posts["text"].iloc[va], labeled_embeddings[va], labeled_tokens, va)
    validation_runs.append((targets[va].numpy(), probs.argmax(1), final_labels(probs, tr)))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### 3.4 Measure the decision adjustment

Compare macro F1 before and after the score adjustments. For the adjusted predictions, report precision, recall, and F1 for each emotion, along with a confusion matrix. For example, the disgust row and anger column count how many disgust posts were labeled as anger.

In [10]:
metrics = []
for fold, (truth, plain, boosted) in enumerate(validation_runs, start=1):
    metrics.append({"fold": fold, "plain": f1_score(truth, plain, average="macro"),
                    "selected": f1_score(truth, boosted, average="macro")})
scores = pd.DataFrame(metrics).set_index("fold")
print(scores.round(4).to_string())
print("Mean macro F1:", scores.mean().round(4).to_dict())
truth = np.concatenate([row[0] for row in validation_runs])
prediction = np.concatenate([row[2] for row in validation_runs])
print(classification_report(truth, prediction, target_names=class_names, digits=4, zero_division=0))
matrix = pd.DataFrame(confusion_matrix(truth, prediction), index=class_names, columns=class_names)
print("Confusion matrix (rows are actual emotions)")
print(matrix.to_string())

       plain  selected
fold                  
1     0.6092    0.6107
Mean macro F1: {'plain': 0.6092, 'selected': 0.6107}
              precision    recall  f1-score   support

       anger     0.6178    0.7185    0.6644       135
     disgust     0.5526    0.4615    0.5030        91
        fear     0.7073    0.5000    0.5859        58
         joy     0.7120    0.7771    0.7432       175
     sadness     0.5821    0.6190    0.6000       126
    surprise     0.6216    0.5227    0.5679        88

    accuracy                         0.6360       673
   macro avg     0.6323    0.5998    0.6107       673
weighted avg     0.6350    0.6360    0.6316       673

Confusion matrix (rows are actual emotions)
          anger  disgust  fear  joy  sadness  surprise
anger        97       13     4    7        9         5
disgust      24       42     3    9        9         4
fear          3        3    29    5       16         2
joy          12        2     2  136       11        12
sadness      16 

#### Final grouped validation audit

Fold 1 was used for development. Across the original five grouped folds, the best mean used equal parts direct MiniLM, SemEval-warmed MiniLM and the lexical classifier. Each fine-tune used seed 42. At prediction time each score is divided by its training-class frequency raised to 0.25, then the disgust score is multiplied by 1.10.

| Fold | Original one-seed approach | Selected approach |
| --- | ---: | ---: |
| 1 | 0.6001 | 0.6107 |
| 2 | 0.5987 | 0.5989 |
| 3 | 0.5983 | 0.6063 |
| 4 | 0.5572 | 0.5663 |
| 5 | 0.5722 | 0.5658 |
| Mean | 0.5853 | 0.5896 |

Disgust recall changed from 156/454 to 167/454. Disgust classified as anger changed from 155 to 140. Fear recall changed from 155/288 to 160/288.

Twelve combinations of models and score adjustments were compared. Choosing the combination on four folds and scoring it on the remaining fold averaged 0.5850. The small gain from the approach chosen using all five folds is uncertain. The leaderboard target remains 0.6424.

#### Comparison across all five grouped folds

Each labeled post was held out once. These are the mean validation scores from the five runs. For each run, the classifiers and TF-IDF vocabularies were fitted on its training fold.

| Model | Macro F1 | Weighted F1 | Micro F1 |
| --- | ---: | ---: | ---: |
| Frozen MiniLM + SVM | 0.5106 | 0.5319 | 0.5405 |
| TF-IDF + frozen MiniLM SVM | 0.5457 | 0.5616 | 0.5676 |
| TF-IDF + frozen MiniLM logistic regression | 0.5328 | 0.5495 | 0.5551 |
| Fine-tuned MiniLM | 0.5730 | 0.5937 | 0.6030 |
| Combined probabilities | **0.5874** | **0.6070** | **0.6149** |

### 3.5 Refit the selected models on all labeled posts

In [11]:
all_rows = np.arange(len(train_posts))
test_tokens = tokenize_posts(test_posts["text"])
test_scores = predict_split(all_rows, test_posts["text"], test_embeddings, test_tokens, np.arange(len(test_posts)))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### 3.6 Export the candidate

Write one emotion label for each of the 1,440 test posts to `submission.csv`, using the columns `id` and `label` in sample-submission order. Running this cell again replaces the existing file.

In [12]:
submission = test_posts[["post_id"]].rename(columns={"post_id": "id"})
submission["label"] = label_encoder.inverse_transform(final_labels(test_scores, all_rows))
assert submission["id"].is_unique and submission["id"].equals(submission_ids["post_id"])
assert submission["label"].isin(class_names).all()
submission.to_csv("submission.csv", index=False)
print(f"Saved {len(submission):,} predictions to submission.csv.")

Saved 1,440 predictions to submission.csv.
